# Targeted SHAP reliability check
Frozen full 3s fused model. Fixed diagnostic: the four original failed detection cases, 2048 and 8192 draws, seeds 730+j and 830+j (failed-case order j). Each case is called separately for nested sampling across budgets. Total 81,920 path draws, no adaptive extension or tolerance changes. Existing backgrounds and validation IDs remain fixed. Magnitude is assessed from saved results only. All executable project code, including the execution harness, is inside this notebook. No prior output is overwritten.

In [1]:
from pathlib import Path
import os,sys
ROOT=next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file())
OUT=ROOT/'explainability/shap_reliability_v1';OLD=ROOT/'explainability/final_attention_shap_v1'
sys.dont_write_bytecode=True;os.environ['PYTHONDONTWRITEBYTECODE']='1'
for key,folder in [('IPYTHONDIR','ipython'),('JUPYTER_RUNTIME_DIR','jupyter_runtime'),('MPLCONFIGDIR','matplotlib_config'),('NUMBA_CACHE_DIR','numba_cache')]:os.environ[key]=str(OUT/folder)
if '--execute-reliability' in sys.argv:
 runtime=ROOT/'cnn/cnn wave/.notebook-runtime';sys.path.insert(0,str(runtime))
 os.environ['PYTHONPATH']=os.pathsep.join([str(OLD/'runtime'),str(runtime)])
 import nbformat
 from nbclient import NotebookClient
 p=OUT/'01_reliability_check.ipynb';nb=nbformat.read(p,as_version=4)
 def progress(cell,cell_index,**kwargs):
  print('Completed cell',cell_index,flush=True)
  for o in cell.get('outputs',[]):
   if o.get('output_type')=='stream':print(o.get('text',''),flush=True)
  nbformat.write(nb,p)
 client=NotebookClient(nb,timeout=1200,kernel_name='python3',resources={'metadata':{'path':str(OUT)}},on_cell_executed=progress)
 try:client.execute()
 finally:nbformat.write(nb,p)
 print('Executed notebook saved.',flush=True)
else:print('Output/cache paths restricted to',OUT)


Output/cache paths restricted to C:\Users\choud\Desktop\Minor project\explainability\shap_reliability_v1


In [2]:
import json,ast,hashlib,inspect,time
sys.path.insert(0,str(OLD/'runtime'))
import numpy as np,torch,h5py,shap
from torch import nn
from scipy.stats import spearmanr
torch.set_num_threads(4);torch.backends.cudnn.enabled=False;torch.backends.mkldnn.enabled=False
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for q in iter(lambda:f.read(4194304),b''):h.update(q)
 return h.hexdigest()
old_paths=[OLD/n for n in ['01_final_attention_shap.ipynb','shap_validation_report.md','shap_explainability_report.md','selection_manifest.json','detection_shap_values.npz','magnitude_shap_values.npz']]
old_hashes={str(p):sha(p) for p in old_paths}
report=(OLD/'shap_validation_report.md').read_text(encoding='utf-8')
prior=json.JSONDecoder().raw_decode(report[report.index('{'):])[0]
source=json.loads((OLD/'01_final_attention_shap.ipynb').read_text(encoding='utf-8'))
manifest=json.loads((OLD/'selection_manifest.json').read_text())
saved={}
for task in ['detection','magnitude']:
 with np.load(OLD/f'{task}_shap_values.npz',allow_pickle=True) as f:saved[task]={k:f[k].copy() for k in f.files}
d=saved['detection'];tol=.1+.1*np.abs(d['prediction']-float(d['expected']))
total=d['waveform'].sum(1)+d['tabular'].sum((1,2))
residual=d['prediction']-float(d['expected'])-total
assert np.allclose(residual,d['residual'],atol=1e-10)
failed=np.flatnonzero(np.abs(residual)>tol);assert len(failed)==4
case_rows=[]
for i in failed:
 case_rows.append(dict(index=int(i),source_id=str(d['source_ids'][i]),logit=float(d['prediction'][i]),expected=float(d['expected']),waveform_sum=float(d['waveform'][i].sum()),tabular_sum=float(d['tabular'][i].sum()),residual=float(residual[i]),tolerance=float(tol[i])))
print('Four original failures:',json.dumps(case_rows,indent=2))


C:\Users\choud\scoop\apps\python313\current\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Four original failures: [
  {
    "index": 2,
    "source_id": "NOISE_P7567",
    "logit": -0.5278972387313843,
    "expected": -0.7625274658203125,
    "waveform_sum": -1.0118956034058468,
    "tabular_sum": 1.112619024067801,
    "residual": 0.13390680642697395,
    "tolerance": 0.12346302717924118
  },
  {
    "index": 5,
    "source_id": "NCS_EV_03052",
    "logit": -0.761950671672821,
    "expected": -0.7625274658203125,
    "waveform_sum": -0.8742322634679625,
    "tabular_sum": 1.0047369789634586,
    "residual": -0.12992792134800468,
    "tolerance": 0.1000576838850975
  },
  {
    "index": 10,
    "source_id": "NCS_EV_07170",
    "logit": -0.0037793111987411976,
    "expected": -0.7625274658203125,
    "waveform_sum": -0.6832957851040592,
    "tabular_sum": 1.817967189514256,
    "residual": -0.3759232306965128,
    "tolerance": 0.17587482929229736
  },
  {
    "index": 11,
    "source_id": "NCS_EV_02345",
    "logit": 0.7256655097007751,
    "expected": -0.7625274658203125,
 

In [3]:
# Exact reused definitions only: no top-level original code or previous audits.
V3=ROOT/'cnn/cnn wave/eew_causal_v3';CR=V3/'cnn_eew_training';BR=ROOT/'bilstm_eew/bilstm_training_v1'
AP=ROOT/'fusion/cnn_bilstm_attention_v1';TAB=ROOT/'bilstm_eew/causal_tabular_v1/tabular_features.h5'
tree=ast.parse(''.join(source['cells'][1]['source']))
for name in ['original_class','FullModel','Scalar']:
 node=next(n for n in tree.body if isinstance(n,(ast.ClassDef,ast.FunctionDef)) and n.name==name)
 exec(compile(ast.Module(body=[node],type_ignores=[]),'original_shap_definition','exec'),globals())
original_class(CR/'01_cnn_training.ipynb',1,'EarlyCNN')
original_class(BR/'01_bilstm_training.ipynb',0,'TabularBiLSTM')
original_class(AP/'01_attention_training.ipynb',0,'AttentionHead')
model_paths=[AP/'models/attention_3s_best.pt',CR/'models/cnn_3s_best.pt',BR/'models/bilstm_3s_best.pt']
model_hashes={str(p):sha(p) for p in model_paths}
assert all(v==prior['input_sha256'][p] for p,v in model_hashes.items())
ck,cc,bc=[torch.load(p,map_location='cpu',weights_only=False) for p in model_paths]
assert ck['horizon']==cc['horizon_seconds']==bc['horizon']==3
c=EarlyCNN();b=TabularBiLSTM();head=AttentionHead()
c.load_state_dict(cc['state_dict'],strict=True);b.load_state_dict(bc['state_dict'],strict=True);head.load_state_dict(ck['head_state'],strict=True)
full=FullModel().eval().requires_grad_(False)
models={'detection':Scalar(0).eval(),'magnitude':Scalar(1).eval()}
initial={k:v.clone() for k,v in full.state_dict().items()}
def selected_inputs(split,ids):
 with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as w,h5py.File(TAB,'r') as t:
  g=t[split];wi=w['source_id'].asstr()[:];wp=w['parent_id'].asstr()[:];tp=g['parent_id'].asstr()[:]
  assert len(set(wi))==len(wi) and len(set(tp))==len(tp)
  wl={v:i for i,v in enumerate(wi)};tl={v:i for i,v in enumerate(tp)}
  a=np.array([wl[v] for v in ids]);z=np.array([tl[wp[i]] for i in a])
  for key in ['source_id','parent_id','labels','magnitude','classification_mask_3s','magnitude_mask_3s','pre_s_valid_3s']:
   wa=w[key].asstr()[:] if w[key].dtype.kind=='O' else w[key][:]
   ta=g[key].asstr()[:] if g[key].dtype.kind=='O' else g[key][:]
   assert np.array_equal(wa[a],ta[z]),key
  assert np.all(w['classification_mask_3s'][:][a])
  assert w.attrs['sampling_rate_hz']==t.attrs['sampling_rate_hz']==100
  assert w.attrs['pre_context_samples']==t.attrs['pre_reference_samples']==100
  assert g['3s'].attrs['last_bin_end_relative_sample_exclusive']==300
  x=torch.from_numpy(np.stack([w['windows_3s_norm'][int(i)] for i in a]))
  v=torch.from_numpy(np.stack([g['3s/X'][int(i)] for i in z]))
  assert x.shape==(len(ids),400) and v.shape==(len(ids),40,7)
  assert torch.isfinite(x).all() and torch.isfinite(v).all()
  return dict(inputs=[x,v],parents=wp[a],labels=w['labels'][:][a],magnitude=w['magnitude'][:][a],wave_rows=a,tab_rows=z)
primary=selected_inputs('train',manifest['background_ids']);alternative=selected_inputs('train',manifest['alternative_background_ids'])
validation=selected_inputs('validation',manifest['validation_ids']['detection'])
assert not((set(primary['parents'])|set(alternative['parents']))&set(validation['parents']))
assert np.array_equal(d['source_ids'],manifest['validation_ids']['detection'])
with torch.no_grad():
 pred=models['detection'](*validation['inputs']).numpy().ravel()
 bg_logits=models['detection'](*primary['inputs']).numpy().ravel()
 alt_logits=models['detection'](*alternative['inputs']).numpy().ravel()
assert np.allclose(pred,d['prediction'],rtol=0,atol=1e-5)
assert abs(float(bg_logits.mean())-float(d['expected']))<1e-6
assert abs(float(alt_logits.mean())-float(d['alternative_expected']))<1e-6
for row in case_rows:
 i=row['index'];row.update(label=int(validation['labels'][i]),magnitude=float(validation['magnitude'][i]),wave_row=int(validation['wave_rows'][i]),tabular_row=int(validation['tab_rows'][i]))
from shap.explainers._gradient import _PyTorchGradient
sampling_source=inspect.getsource(_PyTorchGradient.shap_values)
assert 'np.random.choice(self.data[0].shape[0])' in sampling_source and 'np.random.uniform()' in sampling_source and 'np.random.seed(rseed)' in sampling_source
print('Prediction max error:',float(np.abs(pred-d['prediction']).max()))
print('Expected mean logits:',float(bg_logits.mean()),float(alt_logits.mean()))
print('Paired identity/label/mask checks passed for two 48-row backgrounds and 12 validation records. No test access or full-validation audit.')


Prediction max error: 0.0
Expected mean logits: -0.7625274658203125 1.2113803625106812
Paired identity/label/mask checks passed for two 48-row backgrounds and 12 validation records. No test access or full-validation audit.


In [4]:
stability={}
def compare(a,b):
 fa=np.abs(a[1]).mean((0,1));fb=np.abs(b[1]).mean((0,1))
 aa=np.concatenate([x.reshape(len(x),-1) for x in a],1);bb=np.concatenate([x.reshape(len(x),-1) for x in b],1)
 cos=(aa*bb).sum(1)/(np.linalg.norm(aa,axis=1)*np.linalg.norm(bb,axis=1))
 return dict(feature_rank_spearman=float(spearmanr(fa,fb).statistic),feature_relative_L1=float(np.abs(fa-fb).sum()/fa.sum()),local_cosine_min=float(cos.min()),local_cosine_mean=float(cos.mean()),waveform_temporal_spearman=float(spearmanr(np.abs(a[0]).mean(0),np.abs(b[0]).mean(0)).statistic),tabular_temporal_spearman=float(spearmanr(np.abs(a[1]).mean((0,2)),np.abs(b[1]).mean((0,2))).statistic))
for task,s in saved.items():
 a=[s['waveform'],s['tabular']];repeat=[s['repeat_waveform'],s['repeat_tabular']];other=[s['alternative_waveform'],s['alternative_tabular']]
 repeat_res=s['prediction']-float(s['expected'])-repeat[0].sum(1)-repeat[1].sum((1,2))
 alt_res=s['alternative_prediction']-float(s['alternative_expected'])-other[0].sum(1)-other[1].sum((1,2))
 assert np.allclose(alt_res,s['alternative_residual'],atol=1e-10)
 t=.1+.1*np.abs(s['prediction']-float(s['expected']))
 at=.1+.1*np.abs(s['alternative_prediction']-float(s['alternative_expected']))
 stability[task]=dict(same_background_repeat=compare(a,repeat),alternative_background=compare(a,other),expected_shift=float(s['alternative_expected']-s['expected']),primary_passes=int((np.abs(s['residual'])<=t).sum()),repeat_passes=int((np.abs(repeat_res)<=t).sum()),alternative_passes=int((np.abs(alt_res)<=at).sum()),n=len(s['prediction']))
print(json.dumps(stability,indent=2))


{
  "detection": {
    "same_background_repeat": {
      "feature_rank_spearman": 1.0,
      "feature_relative_L1": 0.007546031055485575,
      "local_cosine_min": 0.9913747632126321,
      "local_cosine_mean": 0.9970651589377323,
      "waveform_temporal_spearman": 0.9979828623928899,
      "tabular_temporal_spearman": 0.9953095684803004
    },
    "alternative_background": {
      "feature_rank_spearman": 0.8928571428571429,
      "feature_relative_L1": 0.13649307648556994,
      "local_cosine_min": 0.9304429567599237,
      "local_cosine_mean": 0.9489614995645762,
      "waveform_temporal_spearman": 0.9543852774079836,
      "tabular_temporal_spearman": 0.8714821763602253
    },
    "expected_shift": 1.9739078283309937,
    "primary_passes": 8,
    "repeat_passes": 10,
    "alternative_passes": 9,
    "n": 12
  },
  "magnitude": {
    "same_background_repeat": {
      "feature_rank_spearman": 1.0,
      "feature_relative_L1": 0.005630774521452867,
      "local_cosine_min": 0.9954507

In [5]:
diagnostic=[];diagnostic_arrays={}
explainer=shap.GradientExplainer(models['detection'],primary['inputs'],batch_size=64,local_smoothing=0)
for j,i in enumerate(failed):
 for seed in [730+j,830+j]:
  for n in [2048,8192]:
   start=time.time()
   raw=explainer.shap_values([v[i:i+1] for v in validation['inputs']],nsamples=n,rseed=seed)
   assert isinstance(raw,list) and len(raw)==2
   a=[np.asarray(raw[0])[...,0],np.asarray(raw[1])[...,0]]
   assert a[0].shape==(1,400) and a[1].shape==(1,40,7) and all(np.isfinite(v).all() for v in a)
   contribution=float(a[0].sum()+a[1].sum())
   # Replay sampler indices only; all attributions above are actual SHAP.
   rng=np.random.RandomState(seed);draws=[]
   for k in range(n):
    draws.append(rng.choice(len(bg_logits)));rng.uniform()
   sampled_base=float(bg_logits[draws].astype('float64').mean());full_base=float(bg_logits.mean())
   r=float(pred[i])-full_base-contribution;reference_error=sampled_base-full_base
   integration_error=float(pred[i])-sampled_base-contribution
   assert abs(r-reference_error-integration_error)<1e-10
   row=dict(source_id=str(d['source_ids'][i]),seed=seed,nsamples=n,residual=r,abs_residual=abs(r),tolerance=float(tol[i]),passes=bool(abs(r)<=tol[i]),sampled_baseline_error=reference_error,remaining_path_error=integration_error,seconds=time.time()-start)
   diagnostic.append(row);diagnostic_arrays[f'case{j}_seed{seed}_n{n}_waveform']=a[0];diagnostic_arrays[f'case{j}_seed{seed}_n{n}_tabular']=a[1]
   print(json.dumps(row),flush=True)
np.savez_compressed(OUT/'diagnostic_attributions.npz',**diagnostic_arrays)


{"source_id": "NOISE_P7567", "seed": 730, "nsamples": 2048, "residual": 0.3703911686706336, "abs_residual": 0.3703911686706336, "tolerance": 0.12346302717924118, "passes": false, "sampled_baseline_error": 0.0433938960291016, "remaining_path_error": 0.326997272641532, "seconds": 2.6880404949188232}


{"source_id": "NOISE_P7567", "seed": 730, "nsamples": 8192, "residual": 0.20842467245411012, "abs_residual": 0.20842467245411012, "tolerance": 0.12346302717924118, "passes": false, "sampled_baseline_error": 0.031551898651571264, "remaining_path_error": 0.17687277380253885, "seconds": 8.923234939575195}


{"source_id": "NOISE_P7567", "seed": 830, "nsamples": 2048, "residual": 0.01782786490203181, "abs_residual": 0.01782786490203181, "tolerance": 0.12346302717924118, "passes": true, "sampled_baseline_error": 0.015989613677220404, "remaining_path_error": 0.0018382512248114047, "seconds": 2.1689865589141846}


{"source_id": "NOISE_P7567", "seed": 830, "nsamples": 8192, "residual": -0.11180235961637353, "abs_residual": 0.11180235961637353, "tolerance": 0.12346302717924118, "passes": true, "sampled_baseline_error": -0.08460220731046775, "remaining_path_error": -0.027200152305905778, "seconds": 8.847530603408813}


{"source_id": "NCS_EV_03052", "seed": 731, "nsamples": 2048, "residual": -0.17215768459451297, "abs_residual": 0.17215768459451297, "tolerance": 0.1000576838850975, "passes": false, "sampled_baseline_error": -0.20991017700816883, "remaining_path_error": 0.03775249241365586, "seconds": 2.455190420150757}


{"source_id": "NCS_EV_03052", "seed": 731, "nsamples": 8192, "residual": 0.18689000907316378, "abs_residual": 0.18689000907316378, "tolerance": 0.1000576838850975, "passes": false, "sampled_baseline_error": 0.01577830915175582, "remaining_path_error": 0.17111169992140796, "seconds": 8.715800046920776}


... (6 outputs left out)

{"source_id": "NCS_EV_02345", "seed": 733, "nsamples": 2048, "residual": 0.23701390962485092, "abs_residual": 0.23701390962485092, "tolerance": 0.24881932139396667, "passes": true, "sampled_baseline_error": -0.055300940300185175, "remaining_path_error": 0.2923148499250361, "seconds": 2.215043544769287}


{"source_id": "NCS_EV_02345", "seed": 733, "nsamples": 8192, "residual": 0.15204338473696044, "abs_residual": 0.15204338473696044, "tolerance": 0.24881932139396667, "passes": true, "sampled_baseline_error": 0.03729972218036437, "remaining_path_error": 0.11474366255659607, "seconds": 8.726908206939697}


{"source_id": "NCS_EV_02345", "seed": 833, "nsamples": 2048, "residual": -0.06646594238136316, "abs_residual": 0.06646594238136316, "tolerance": 0.24881932139396667, "passes": true, "sampled_baseline_error": -0.02977027345514216, "remaining_path_error": -0.036695668926221, "seconds": 2.0937156677246094}


{"source_id": "NCS_EV_02345", "seed": 833, "nsamples": 8192, "residual": 0.010940026925205126, "abs_residual": 0.010940026925205126, "tolerance": 0.24881932139396667, "passes": true, "sampled_baseline_error": -0.011686121308969177, "remaining_path_error": 0.022626148234174304, "seconds": 9.09932541847229}


In [6]:
assert all(torch.equal(initial[k],v) for k,v in full.state_dict().items())
assert all(not p.requires_grad and p.grad is None for p in full.parameters())
assert all(not m.training for m in full.modules())
assert all(sha(Path(p))==v for p,v in old_hashes.items())
assert all(sha(Path(p))==v for p,v in model_hashes.items())
summary={}
for n in [2048,8192]:
 rows=[r for r in diagnostic if r['nsamples']==n]
 summary[n]=dict(mean_absolute_residual=float(np.mean([r['abs_residual'] for r in rows])),rms_residual=float(np.sqrt(np.mean([r['residual']**2 for r in rows]))),max_absolute_residual=max(r['abs_residual'] for r in rows),passes=sum(r['passes'] for r in rows),runs=len(rows),mean_absolute_background_sampling_error=float(np.mean([abs(r['sampled_baseline_error']) for r in rows])),mean_absolute_remaining_path_error=float(np.mean([abs(r['remaining_path_error']) for r in rows])))
improved=sum(diagnostic[k+1]['abs_residual']<diagnostic[k]['abs_residual'] for k in range(0,len(diagnostic),2))
results=dict(verdict='CONDITIONAL',failed_original_cases=case_rows,sampling_summary=summary,paired_seed_runs_improved=improved,diagnostic=diagnostic,existing_stability=stability,original_files_unchanged=True,weights_unchanged=True,budget_path_draws=sum(r['nsamples'] for r in diagnostic),versions={'torch':str(torch.__version__),'shap':shap.__version__},source_sha256=old_hashes,checkpoint_sha256=model_hashes)
(OUT/'reliability_results.json').write_text(json.dumps(results,indent=2),encoding='utf-8')
def table(headers,rows):
 return '\n'.join(['| '+' | '.join(headers)+' |','|'+'|'.join(['---']*len(headers))+'|']+['| '+' | '.join(map(str,r))+' |' for r in rows])
failure_table=table(['Validation source ID','Label','Magnitude','Logit','Wave sum','Tabular sum','Residual','Tolerance'],[[r['source_id'],r['label'],f"{r['magnitude']:.2f}",f"{r['logit']:.6f}",f"{r['waveform_sum']:.6f}",f"{r['tabular_sum']:.6f}",f"{r['residual']:.6f}",f"{r['tolerance']:.6f}"] for r in case_rows])
sampling_table=table(['Draws','Mean absolute residual','RMS residual','Maximum residual','Passes / runs','Mean absolute baseline sampling error','Mean absolute remaining path error'],[[n,f"{s['mean_absolute_residual']:.6f}",f"{s['rms_residual']:.6f}",f"{s['max_absolute_residual']:.6f}",f"{s['passes']}/{s['runs']}",f"{s['mean_absolute_background_sampling_error']:.6f}",f"{s['mean_absolute_remaining_path_error']:.6f}"] for n,s in summary.items()])
detail_table=table(['Case','Seed','Draws','Signed residual','Tolerance','Pass'],[[r['source_id'],r['seed'],r['nsamples'],f"{r['residual']:.6f}",f"{r['tolerance']:.6f}",r['passes']] for r in diagnostic])
decreased=summary[8192]['mean_absolute_residual']<summary[2048]['mean_absolute_residual']
finding=('Mean absolute reconstruction error decreased' if decreased else 'Mean absolute reconstruction error did not decrease')+f" from {summary[2048]['mean_absolute_residual']:.6f} to {summary[8192]['mean_absolute_residual']:.6f} logits; {improved}/8 matched case-seed comparisons improved. These are four deliberately selected failures and two seeds, not a convergence guarantee. No post-result budget extension or tolerance adjustment was made."
dependency='Passing reconstruction establishes only approximate numerical accounting for the chosen model function and reference distribution. It does not validate the counterfactual inputs. Waveform and seven-feature tensors share a source, and paired real endpoints do not make straight-line interpolated points obey nonlinear feature extraction (including energy/RMS dependence). Consequently these are expected-gradient model-input attributions, not uniquely identifiable physical branch contributions, conditional-distribution SHAP, independent sensor effects, or causal effects. Attention pools bidirectional hidden states and remains distinct from SHAP; a bin cannot be interpreted as isolated physical evidence.'
lines=['# Targeted SHAP reliability report','','**FINAL VERDICT: CONDITIONAL**','','## Four original detection failures','',failure_table,'','## Implementation checks','','The original residual is logit minus the empirical mean of 48 training-background logits minus the signed sum over all 400 waveform and 40x7 tabular SHAP values. No sigmoid, probability/logit mixing, absolute-value summation, or branch averaging was found. The original scalar wrapper returns shape (N,1) from the full fused model. SHAP singleton output axes are removed before aggregation. Recomputed saved-array residuals agree to 1e-10; reloaded frozen predictions agree to 1e-5 and background means to 1e-6. Expected values are mean(model(background)), not model(mean(background)) or logit(mean probability).','','The unchanged rule abs(residual) <= 0.1 + 0.1*abs(logit - expected logit) is a prespecified heuristic screen in logit units. It is neither SHAP exact local accuracy nor a statistical confidence bound. It is reference-dependent and has an absolute floor, so passing fractions alone are not adequate reliability evidence. Signed residuals and tolerances are shown rather than concealing failures.','','## Fixed-budget sampling diagnostic','','Two nested sampling budgets, 2048 and 8192, two seeds per failed case (730+j, 830+j), four cases, 81,920 total path draws. Primary training background unchanged; every case is called separately so matching seeds use the same random-sample prefix. These newly seeded runs are not exact continuations of the original batched seed-720 run. Only GradientExplainer attributions are used.','',sampling_table,'',finding,'',detail_table,'','The residual was partitioned by replaying the installed SHAP sampler: sampled-background mean minus full-background mean, plus logit minus sampled-background mean minus summed attribution. The second term includes finite path-integration/gradient-sampling and floating-point error; the partition is not a formal variance decomposition or proof of the sole cause. SHAP draws the same background row and interpolation coefficient for both branches.','','## Existing background and repeat stability','','Detection expected logit changes from -0.762527 to 1.211380 (+1.973908). Magnitude expected output changes from 3.853153 to 3.861201 (+0.008048). Same coarse stratum counts do not guarantee similar background output distributions. Feature rankings alone therefore cannot justify reference-invariant local interpretations. No new backgrounds were selected.','']
for task,s in stability.items():
 lines += [f'### {task}', '',table(['Measure','Same-background repeat','Alternative background'],[[key,f"{s['same_background_repeat'][key]:.6f}",f"{s['alternative_background'][key]:.6f}"] for key in s['same_background_repeat']]),'',f"Reconstruction passes: original {s['primary_passes']}/{s['n']}, repeat {s['repeat_passes']}/{s['n']}, alternative {s['alternative_passes']}/{s['n']}.",'']
lines += ['## Paired-input and preservation checks','','Both fixed 48-record training backgrounds and the 12 existing validation records were reloaded by source ID and joined by unique parent ID. Source ID, parent ID, label, magnitude and 3s eligibility masks match across branches; ordered source IDs match the saved attribution arrays. Input shapes are (N,400)/(N,40,7); 100 Hz, one second pre-context, and final bin endpoint 3s are preserved. Only selected waveform/feature rows were loaded. No test group, new validation sample, whole-model audit, training or model selection was used. Existing SHAP sources and checkpoint hashes remain unchanged; model state tensors and evaluation mode are unchanged. No assertion here independently re-audits original feature generation.','','## Remaining scientific limits','',dependency,'','The verdict remains CONDITIONAL even if the larger-budget numerical checks pass: reference sensitivity and dependent-input interpretation remain, while the original figures still contain their original 2048-draw values. This diagnostic does not repair or overwrite them. All results and code are retained in the executed notebook; diagnostic arrays and machine-readable results are new artifacts in this directory.']
(OUT/'reliability_report.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
guidance=['# Academic report integration guidance','','**FINAL VERDICT: CONDITIONAL.** Use existing figures only as descriptive explanations of the frozen model on the preselected validation cohort. No existing plot has been replaced by this diagnostic.','','## Figure-by-figure disposition','',table(['Existing figure','Academic use','Required qualification'],[
['detection_feature_importance.png','Qualified aggregate','12 deliberately stratified cases; approximate mean absolute model-input SHAP; background feature rank rho=0.893; original 4/12 reconstruction failures.'],
['detection_waveform_shap.png','Qualified aggregate','Not a physical time-local causal signal; original 4/12 reconstruction failures and 1.974-logit reference shift.'],
['detection_temporal_importance.png','Qualified aggregate','Sum of absolute feature attributions per bin, not signed evidence or Attention; original numerical limitations apply.'],
['detection_feature_time_heatmap.png','Qualified aggregate','Mean absolute SHAP hides sign/record variation; dependent inputs; original numerical and reference-sensitivity warning required.'],
['detection_local_explanation.png','Usable as an explicitly approximate local explanation','NCS_EV_03910 passes the original tolerance (residual 0.201782 < tolerance about 0.25271). Retain its residual and reference-sensitivity warning; do not present an exact logit or probability decomposition.'],
['detection_attention_vs_shap.png','Methodological comparison with explicit warning','Attention and SHAP differ in units/meaning; resemblance is not validation or causality; original detection reconstruction limits apply.'],
['magnitude_feature_importance.png','Usable with study limitations','Nine eligible earthquakes; strong events deliberately included; mixed-class mathematical background; not population importance.'],
['magnitude_waveform_shap.png','Usable with study limitations','Model-input attributions only; no physical causality despite 9/9 original reconstruction passes.'],
['magnitude_temporal_importance.png','Usable with study limitations','Absolute attribution across seven inputs; bidirectional context spans the full observed window.'],
['magnitude_feature_time_heatmap.png','Usable with study limitations','Dependent normalized features; mean absolute values do not show direction.'],
['magnitude_local_explanation.png','Approximate local model explanation','Retain residual and predicted/true magnitude; numerical pass does not establish prediction accuracy or physical validity.'],
['magnitude_attention_vs_shap.png','Methodological comparison with explicit warning','Attention is not SHAP; weights cannot validate SHAP and axes are not directly comparable.']
]),'','All filenames refer to ../final_attention_shap_v1/plots/. Existing 300-DPI images remain unchanged. Magnitude plots have stronger numerical reconstruction support, but still require dependency, sampling and synthetic oracle-alignment limitations. Detection aggregate plots require explicit numerical and reference-sensitivity warnings, not merely a generic SHAP caveat.','','## Suggested academic text','','A targeted fixed-budget check examined the four detection records that failed the original SHAP reconstruction tolerance. '+finding+' The frozen predictions, paired inputs, logit handling, empirical expected value and signed multi-input aggregation were verified. The detection reference logit differed by 1.974 between two existing training backgrounds despite relatively stable feature rankings. Detection explanations therefore remain descriptive and approximate; the original local detection figure is presented only with its reconstruction residual. Magnitude reconstruction passed for all nine original earthquake cases, without establishing causal or independent-modality interpretations.','','## Required shared limitations','',dependency,'','Do not caption any figure as a physical causal contribution, independent modality importance, exact probability decomposition, real-world warning validation, or proof that Attention agrees with SHAP. Do not attach new high-sampling diagnostic values to unchanged old figures as if those figures were regenerated.']
(OUT/'report_integration_guidance.md').write_text('\n'.join(guidance)+'\n',encoding='utf-8')
print(json.dumps(summary,indent=2));print(finding);print('CONDITIONAL. Reports saved; originals unchanged.')


{
  "2048": {
    "mean_absolute_residual": 0.21209244775641933,
    "rms_residual": 0.24573574584028257,
    "max_absolute_residual": 0.38177588943639185,
    "passes": 4,
    "runs": 8,
    "mean_absolute_background_sampling_error": 0.09790319065359654,
    "mean_absolute_remaining_path_error": 0.13745261528128305
  },
  "8192": {
    "mean_absolute_residual": 0.11912039216358863,
    "rms_residual": 0.14458268666822535,
    "max_absolute_residual": 0.22667753946214342,
    "passes": 5,
    "runs": 8,
    "mean_absolute_background_sampling_error": 0.041866153796100036,
    "mean_absolute_remaining_path_error": 0.08107889260501541
  }
}
Mean absolute reconstruction error decreased from 0.212092 to 0.119120 logits; 6/8 matched case-seed comparisons improved. These are four deliberately selected failures and two seeds, not a convergence guarantee. No post-result budget extension or tolerance adjustment was made.
CONDITIONAL. Reports saved; originals unchanged.
